In [11]:
# Carga rápida y selección de columnas usando settings y fallback local

import sys
import re
from pathlib import Path
import io
import asyncio
import pandas as pd
import numpy as np

# 1. Resolver ruta hacia 'src' para importar módulos del proyecto
CURRENT_DIR = Path.cwd().resolve()
SRC_PATH = str(CURRENT_DIR.parent) if CURRENT_DIR.name == "notebooks" else str(CURRENT_DIR / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from config.settings import settings
from dependencies.dependencies import get_storage_repository

COLS = [
    "Date", "Stock_symbol", "Raw_Stock_symbol", 
    "Article_title", "Url", "Metodo_Captura"
]

# 2. Comprobar si existe archivo local en src/data/ o descargar directamente desde Cloudflare R2 vía Storage Repository
local_candidates = [
    Path("src/data/fnspid_bancos_v2.parquet"),
    Path("../data/fnspid_bancos_v2.parquet"),
    Path("fnspid_bancos_v2.parquet"),
]
ruta_local = next((p for p in local_candidates if p.exists()), None)

if ruta_local:
    print(f"Cargando dataset local desde {ruta_local}...")
    df = pd.read_parquet(ruta_local, columns=COLS)
else:
    print("Cargando dataset directamente desde Cloudflare R2 vía Storage Repository...")
    storage = get_storage_repository()
    R2_KEY = "raw/huggingface/fnspid_bancos_v2.parquet"
    data = asyncio.run(storage.get(R2_KEY))
    df = pd.read_parquet(io.BytesIO(data), columns=COLS)

print(f"Dataset cargado exitosamente: {df.shape[0]:,} registros x {df.shape[1]} columnas.")


Cargando dataset directamente desde Cloudflare R2 vía boto3...
Dataset cargado exitosamente: 540,156 registros x 6 columnas.


In [12]:
# Análisis de fechas y rescate por URL 
# Busca los nulos, extrae fechas ocultas en los enlaces y recorta el dataset estrictamente a la ventana 2010–2023.

# 1. Análisis inicial de fechas
df["Date_clean"] = df["Date"].replace({"nan": np.nan, "None": np.nan, "": np.nan})
df["dt_parsed"] = pd.to_datetime(df["Date_clean"], utc=True, errors="coerce")

nulos_iniciales = df["dt_parsed"].isna().sum()
print(f"Nulos iniciales en fecha: {nulos_iniciales:,}")

# 2. Rescate por URL: En caso de nulos, intenta extraer la fecha de la URL.
mask_na = df["dt_parsed"].isna()
if mask_na.sum() > 0:
    fechas_url = (
        df.loc[mask_na, "Url"]
        .str.extract(r"((?:200\d|201\d|202[0-4])[-/]\d{2}[-/]\d{2})")[0]
        .str.replace("/", "-", regex=False)
    )
    df.loc[mask_na, "dt_parsed"] = pd.to_datetime(fechas_url, utc=True, errors="coerce")

nulos_finales = df["dt_parsed"].isna().sum()
print(f"Fechas rescatadas de la URL: {nulos_iniciales - nulos_finales:,}")

# 3. Filtrar ventana del TFM y descartar datos fuera de rango
df_valid = df.dropna(subset=["dt_parsed"]).copy()
df_valid["year"] = df_valid["dt_parsed"].dt.year
df_2010_2023 = df_valid[(df_valid["year"] >= 2010) & (df_valid["year"] <= 2023)].copy()

print(f"Total registros en ventana 2010-2023: {len(df_2010_2023):,}")

Nulos iniciales en fecha: 0
Fechas rescatadas de la URL: 0
Total registros en ventana 2010-2023: 473,136


In [13]:
# Aislamos los registros que quedaron estrictamente fuera de 2010-2023

df_fuera_rango = df_valid[(df_valid["year"] < 2010) | (df_valid["year"] > 2023)]

# Contamos la frecuencia de publicación agrupada por año
distribucion_anios = df_fuera_rango["year"].value_counts().sort_index()
total_descartados = distribucion_anios.sum()

print("=== DISTRIBUCIÓN DE AÑOS DESCARTADOS ===")
print(distribucion_anios)
print("-" * 40)
print(f"Total registros descartados: {total_descartados:,}")

=== DISTRIBUCIÓN DE AÑOS DESCARTADOS ===
year
1999       28
2000       34
2001       30
2002       24
2003       22
2004       24
2005       40
2006       36
2007    11652
2008    36310
2009    18820
Name: count, dtype: int64
----------------------------------------
Total registros descartados: 67,020


In [14]:
# Deduplicación estricta. Elimina duplicados dando prioridad a los etiquetados oficiales (EXACT_TICKER) frente a nuestras capturas.

df_2010_2023["date_day"] = df_2010_2023["dt_parsed"].dt.date
df_2010_2023["title_norm"] = df_2010_2023["Article_title"].str.strip().str.lower()

# Sistema de prioridades
prioridad = {
    "EXACT_TICKER": 1, 
    "DIRTY_TICKER": 2, 
    "RESCUED_NO_TICKER": 3, 
    "CROSS_MENTION_OTHER_TICKER": 4
}
df_2010_2023["prio"] = df_2010_2023["Metodo_Captura"].map(prioridad)

antes_dedup = len(df_2010_2023)
df_2010_2023 = (
    df_2010_2023.sort_values("prio")
    .drop_duplicates(subset=["Stock_symbol", "date_day", "title_norm"], keep="first")
    .drop(columns=["prio", "title_norm"])
)

print(f"Duplicados eliminados: {antes_dedup - len(df_2010_2023):,}")
print(f"Noticias únicas limpias: {len(df_2010_2023):,}")

Duplicados eliminados: 263,024
Noticias únicas limpias: 210,112


In [15]:
# 1. FILTRO ANTI-RUIDO: Eliminamos noticias donde el banco actúa como analista

PATRON_RUIDO = re.compile(
    r"\b(?:upgrades?|downgrades?|initiates?\s+coverage|price\s+target|pt\s+to|"
    r"maintains?\s+(?:buy|hold|neutral|sell|overweight|underweight)|"
    r"reiterates?\s+(?:buy|outperform)|top\s+picks?|stocks?\s+to\s+buy|"
    r"strategists?\s+say|analysts?\s+at|favors?\s+these)\b",
    re.IGNORECASE
)
es_ruido_analista = df_2010_2023["Article_title"].fillna("").str.contains(PATRON_RUIDO, regex=True)
df_sin_analistas = df_2010_2023[~es_ruido_analista].copy()

# 2. WHITELIST ESTRICTA: Separamos los datos oficiales de los rescatados
mask_oficial = df_sin_analistas["Metodo_Captura"].isin(["EXACT_TICKER", "DIRTY_TICKER"])
df_oficial = df_sin_analistas[mask_oficial]
df_rescatado = df_sin_analistas[df_sin_analistas["Metodo_Captura"] == "RESCUED_NO_TICKER"].copy()

# Exigimos que las noticias rescatadas contengan el nombre completo de la entidad
nombres_completos = {
    "JPM": r"\b(jpmorgan|jp morgan)\b",
    "BAC": r"\b(bank of america|bofa)\b",
    "C":   r"\b(citigroup|citibank)\b",
    "GS":  r"\b(goldman sachs)\b",
    "MS":  r"\b(morgan stanley)\b",
    "WFC": r"\b(wells fargo)\b"
}

texto_combinado = (df_rescatado["Article_title"].fillna("") + " " + df_rescatado["Url"].fillna("")).str.lower()
mascara_rescate_valido = pd.Series(False, index=df_rescatado.index)

for ticker, patron in nombres_completos.items():
    es_el_ticker = df_rescatado["Stock_symbol"] == ticker
    tiene_nombre_largo = texto_combinado.str.contains(patron, regex=True)
    mascara_rescate_valido = mascara_rescate_valido | (es_el_ticker & tiene_nombre_largo)

df_rescatado_limpio = df_rescatado[mascara_rescate_valido]

# 3. ENSAMBLAJE FINAL
df_limpio = pd.concat([df_oficial, df_rescatado_limpio])

print(f"1. Noticias de analistas eliminadas: {es_ruido_analista.sum():,}")
print(f"2. Basura eliminada por acrónimos ambiguos (MS, C, etc.): {len(df_rescatado) - len(df_rescatado_limpio):,}")
print("-" * 50)
print(f"DATASET FINAL ESTRICTO Y ÚTIL: {len(df_limpio):,}")


C:\Users\0021824\AppData\Local\Temp\ipykernel_10656\1870346837.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tiene_nombre_largo = texto_combinado.str.contains(patron, regex=True)


1. Noticias de analistas eliminadas: 40,642
2. Basura eliminada por acrónimos ambiguos (MS, C, etc.): 15,984
--------------------------------------------------
DATASET FINAL ESTRICTO Y ÚTIL: 108,209


In [16]:
# Matrices de Cobertura y Diagnóstico Final Muestra la densidad de datos rescatados por año y banco para confirmar si hemos tapado los huecos.

print("=" * 80)
print("NUEVA COBERTURA 2010-2023 (EXACT + RESCUED LIMPIO SIN RUIDO DE ANALISTA)")
print("=" * 80)

pivot = pd.pivot_table(
    df_limpio, index="year", columns="Stock_symbol", values="Article_title", aggfunc="count", fill_value=0
).reindex(range(2010, 2024), fill_value=0)
display(pivot)

print("\n" + "=" * 80)
print("DIAGNÓSTICO DE AÑOS FALTANTES TRAS EL RESCATE")
print("=" * 80)

resumen = []
for banco in ["BAC", "C", "GS", "JPM", "MS", "WFC"]:
    s = pivot[banco] if banco in pivot.columns else pd.Series(0, index=range(2010, 2024))
    ceros = s[s == 0].index.tolist()
    resumen.append({
        "Banco": banco,
        "Total Noticias Limpias": int(s.sum()),
        "Años Cubiertos": int((s > 0).sum()),
        "Años con 0 Noticias": ceros if ceros else "Ninguno"
    })

display(pd.DataFrame(resumen))

NUEVA COBERTURA 2010-2023 (EXACT + RESCUED LIMPIO SIN RUIDO DE ANALISTA)


Stock_symbol,BAC,C,GS,JPM,MS,WFC
year,,,,,,
2010,2130,2114,1129,3985,2679,775
2011,2972,2185,1903,7277,2706,1041
2012,1998,2431,2747,6692,4599,1490
2013,1604,1155,904,2107,1504,1116
2014,1352,1108,751,1044,1288,966
2015,1305,1640,659,827,1437,1121
2016,467,1143,914,338,791,2210
2017,331,1254,945,2,706,1861
2018,214,1012,776,488,657,1010



DIAGNÓSTICO DE AÑOS FALTANTES TRAS EL RESCATE


,Banco,Total Noticias Limpias,Años Cubiertos,Años con 0 Noticias
0,BAC,12831,11,"[2021, 2022, 2023]"
1,C,16875,13,[2021]
2,GS,16560,14,Ninguno
3,JPM,25087,11,"[2021, 2022, 2023]"
4,MS,20671,14,Ninguno
5,WFC,16185,14,Ninguno


In [17]:
# Aislar solo las noticias rescatadas que superaron el filtro anti-ruido
df_rescatadas = df_limpio[df_limpio["Metodo_Captura"] == "RESCUED_NO_TICKER"]

print(f"Total de noticias rescatadas limpias: {len(df_rescatadas):,}")

# Seleccionar columnas clave para que la tabla se lea bien en pantalla
cols_visualizar = ["date_day", "Stock_symbol", "Article_title", "Url"]

# Configurar pandas para que no corte el texto de los titulares largos
pd.set_option('display.max_colwidth', None)

# Mostrar una muestra aleatoria de 10 noticias
display(df_rescatadas[cols_visualizar].sample(10, random_state=42))

Total de noticias rescatadas limpias: 69,138


,date_day,Stock_symbol,Article_title,Url
459510,2014-11-04,MS,REG-Morgan Stanley Griff Sexton Retires from the Board of Directors of Morgan Stanley; Jami Miscik Elected to the Board of Directors,http://www.reuters.com/article/2014/11/03/morgan-stanley-idUSnBw025021a+100+BSW20141103
467895,2011-02-28,JPM,REG - JP Morgan Asian Inv - Net Asset Value(s),http://www.reuters.com/article/2011/02/28/idUS144951+28-Feb-2011+RNS20110228
519095,2012-06-25,JPM,REG - JPMorgan Russ - Net Asset Value(s),http://www.reuters.com/article/2012/06/25/idUS97865+25-Jun-2012+RNS20120625
416633,2012-02-07,GS,Goldman Sachs to buy Dwight asset management: sources,http://www.reuters.com/article/2012/02/07/us-dwight-idUSTRE8161RP20120207
452490,2012-03-12,MS,REG - Morgan Stanley&CoLLCMisys; PLC - Form 8.5 (EPT/RI) Misys Plc,http://www.reuters.com/article/2012/03/12/idUS98405+12-Mar-2012+RNS20120312
468930,2010-02-05,BAC,"UPDATE 1-MOVES-Sagent Advisors, Deutsche Bank, BofA Merrill",http://www.reuters.com/article/etfNews/idUSN0524675320100205
445893,2012-01-13,JPM,REG - JPMorgan Elect PLC - Net Asset Value(s),http://www.reuters.com/article/2012/01/13/idUS101350+13-Jan-2012+RNS20120113
425663,2011-01-12,JPM,REG - JPMorgan Small Co IT - Net Asset Value(s),http://www.reuters.com/article/2011/01/12/idUS127792+12-Jan-2011+RNS20110112
522875,2010-03-25,C,"ProStockMonitor.com Thursday Watch List: Citigroup, Best Buy, Wells Fargo and Bank of America",http://www.reuters.com/article/pressRelease/idUS183499+25-Mar-2010+MW20100325
417542,2013-11-08,MS,"Time Warner Chief Financial & Administrative Officer John Martin to Participate in the Morgan Stanley Technology, Media & Telecoms Conference",http://www.reuters.com/article/2013/11/08/ny-time-warner-idUSnBw085825a+100+BSW20131108


In [18]:
# 1. Patrón específico para detectar "Multiple Sclerosis" camuflado como Morgan Stanley
PATRON_MEDICO_MS = re.compile(
    r"\b(?:multiple\s+sclerosis|pill|drug|treatment|patients?|disease|therapy|biogen|fda|pharma|clinical\s+trials?|dose)\b",
    re.IGNORECASE
)

# 2. Identificar filas donde el banco es MS pero el titular habla de medicina
falsos_positivos_ms = (
    (df_limpio["Stock_symbol"] == "MS") & 
    df_limpio["Article_title"].fillna("").str.contains(PATRON_MEDICO_MS, regex=True)
)

print(f"Falsos positivos de Morgan Stanley (Esclerosis Múltiple) detectados y eliminados: {falsos_positivos_ms.sum():,}")

# 3. Aplicar el filtro para quedarnos solo con lo que NO sea un falso positivo médico
df_limpio = df_limpio[~falsos_positivos_ms]

print(f"NUEVO VOLUMEN DEL DATASET ÚTIL: {len(df_limpio):,}")

Falsos positivos de Morgan Stanley (Esclerosis Múltiple) detectados y eliminados: 137


NUEVO VOLUMEN DEL DATASET ÚTIL: 108,072


In [19]:
# Problemas principales detectados en el dataset final: confusión de acrónimos (MS, C), ruido de analistas, y fechas faltantes. A pesar de los filtros, 
# aún pueden existir casos residuales de confusión de acrónimos o ruido de analistas. Se recomienda una revisión manual adicional para asegurar la calidad del dataset final.